**Fig. 4D and Video S4: whole-tissue run on the pre-placode scale stage with the DSM pair**

Needs a GPU.

In [ ]:
# Working directory: data/fig4/ under the repository root, where the Fig. 4 deposit folder is downloaded
# (domain-specialized_finetuned_bootstrap/ = the DSM model pair, one folder per sample with its
# input volume and the final mask, wound_finetuning/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig4")
print("working directory:", os.getcwd())


# EpiCycle ScaledSegmentation Pipeline

Run 3D cell segmentation, reconstruction, and assembly on tiled image data.

**Usage:** Fill in the paths and parameters below, then Run All.

## 1. Paths & Input

In [ ]:
# ===== PATHS (edit these) =====

# Input Z-stack TIFF (the original microscope file name is in the README) — set this to tile from scratch, or leave None if already tiled
INPUT_TIF = "scale_preplacode/input.tif"   # Set to None if tiles already exist

# Working directory (tiles + outputs go here)
# If INPUT_TIF is set, tiles will be created at WORK_DIR/tiles/
# If INPUT_TIF is None, WORK_DIR should contain existing tile subdirectories
WORK_DIR = "scale_preplacode"

# CellPose model paths (.pth files)
LATERAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True"
AXIAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_lenient_rescale_True"

# Output directory name (created inside WORK_DIR)
OUTPUT_NAME = "Preplacode"

## 2. Resource Configuration

In [ ]:
# ===== RESOURCES (auto-detected from the job's allocation if None) =====

N_GPUS = None            # None = auto-detect from SLURM/CUDA
N_CPU_PER_WORKER = None  # None = auto (total_cores / n_gpus in parallel mode)

## 3. Stage Selection & Tiling Parameters

In [ ]:
# Which stages to run (set False to skip)
RUN_TILING = INPUT_TIF is not None   # Auto: tile if input TIFF provided
# RUN_TILING = False
RUN_SEGMENTATION = True
RUN_RECONSTRUCTION = True
RUN_ASSEMBLY = True 
RUN_GLOBAL_RECONSTRUCTION = True

# ===== TILING (Stage 0) =====
PATCH_SIZE = 1024           # Target XY patch size in pixels
# OVERLAP is shared with assembly (set in Section 6)
OVERLAP = 0.25

Z_THRESHOLD_METHOD = "hard"  # "hard", "percentile", or "default"
Z_THRESHOLD_VALUE = 0.07     # Intensity threshold for Z-range detection
CHANNEL_INDEX = 1            # Which channel for Z-range analysis
AUTO_CORRECT_PATCH = False   # Auto-find valid patch size if exact tiling fails

## 4. Segmentation Parameters

In [ ]:
# ===== SEGMENTATION (Stage 1) =====
# Recorded for reference: the stage is built without segment_params below, so the engine
# defaults were used (channels [2, 1], cell probability 0, flow threshold 0.2). The flow
# threshold acts on the 2D masks only; the 3D reconstruction uses the probability and flow maps.

# GPU allocation strategy: "auto", "fixed" (3-GPU triplets), "round_robin"
GPU_STRATEGY = "auto"

# CellPose parameters
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0
CHANNELS = [2, 1]      # [signal, nucleus] — adjust for your data
BATCH_SIZE = 1024

## 5. Reconstruction Parameters

In [ ]:
# ===== RECONSTRUCTION (Stage 2) =====

# Per-tile direct reconstruction: the engine's u-Segment3D defaults, with gradient renormalisation
# off as in the engine version the runs used (the plugin's current default is on).
RECON_RENORM_GRADIENT = False

# "sequential" (one tile at a time, all cores) or "parallel" (multiple tiles)
RECON_MODE = "parallel"
# RECON_MODE = "sequential"

# Max concurrent tiles in parallel mode (None = n_gpus)
# Rule of thumb: 1900 TB node / ~450 GB per 1920x1920 tile ≈ 4 max
RECON_MAX_WORKERS = None

# Post-processing
SAVE_PROB3D = True
SAVE_FLOW3D = True
SAVE_BINARY = True
BINARY_ERODE = True
BINARY_KERNEL_SIZE = 3
BINARY_BORDER_MARGIN = 2
BINARY_FILL_HOLES = True

# Delete stage1 intermediates after reconstruction to save disk
CLEANUP_INTERMEDIATES = False

## 6. Assembly Parameters

In [ ]:
# ===== ASSEMBLY (Stage 3) =====

# Full volume dimensions in pixels (H, W)
FULL_SIZE = (1792, 1792)

# Tile size and overlap (must match how tiles were cut)
TILE_SIZE = PATCH_SIZE

# Number of Z slices (None = auto-detect from first tile)
DEPTH = None

# Which volumes to stitch
STITCH_MASK = True
STITCH_PROB = True
STITCH_BINARY = True

# Overlap blending for probabilities: "average" or "max"
PROB_STITCH_MODE = "max"

## 7. Global Indirect Reconstruction Parameters

In [ ]:
# ===== GLOBAL INDIRECT RECONSTRUCTION (Stage 4) =====

DOWNSAMPLE_FACTOR = 2          # Downsample stitched volume by this factor
GLOBAL_MIN_SIZE = 1000         # Remove objects smaller than this (voxels, after downsampling)
USE_PRECOMPUTED_BINARY = True  # Use stitched binary mask as foreground constraint

# Gradient descent parameters
# GLOBAL_N_ITER = 30            # Gradient descent iterations (more = finer, slower)
GLOBAL_N_ITER = 300            # Gradient descent iterations (more = finer, slower)
GLOBAL_MOMENTA = 0.9          # Momentum for gradient descent
# GLOBAL_GRADIENT_DECAY = 0.001    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.01    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.1    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.05    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
GLOBAL_GRADIENT_DECAY = 0.25    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower

GLOBAL_RENORM_GRADIENT = True  # Renormalize gradient at each step

# Indirect method parameters
# GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_SMOOTH_SKEL_SIGMA = 1   # Skeleton smoothing sigma
GLOBAL_N_CPU = 64              # CPU threads for multiprocess watershed
GLOBAL_TILE_SHAPE = None       # Tile shape for watershed (None = auto)

## 8. Setup & Pre-flight Check

In [ ]:
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# Resolve GPU count
from packages.epicycle_scaledseg.utils.resources import get_allocated_gpus
n_gpus = N_GPUS or get_allocated_gpus()

work_dir = Path(WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

# Output lives inside the working directory
output_dir = work_dir / OUTPUT_NAME
output_dir.mkdir(exist_ok=True)
log_dir = output_dir / "logs"
log_dir.mkdir(exist_ok=True)

# Tiles directory
tiles_dir = work_dir / "tiles"

if RUN_TILING:
    print(f"Input TIFF: {INPUT_TIF}")
    print(f"Tiles will be created in: {tiles_dir}")
    print(f"Patch size: {PATCH_SIZE}, overlap: {OVERLAP}")
else:
    # Discover existing tile directories
    tile_dirs = sorted([
        str(p) for p in work_dir.iterdir()
        if p.is_dir() and not p.name.startswith(("stage1", "stage2", "stage3", "stage4", "logs", ".", "epicycle"))
        and p.name != OUTPUT_NAME
    ])
    print(f"Found {len(tile_dirs)} existing tile directories in {work_dir}")
    for td in tile_dirs[:5]:
        print(f"  {Path(td).name}")
    if len(tile_dirs) > 5:
        print(f"  ... and {len(tile_dirs) - 5} more")

print(f"\nGPUs: {n_gpus}")
print(f"Output: {output_dir}")

Input TIFF: <run>/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL7.7_PREplacode_002 - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif
Tiles will be created in: <run>/WT_Developing_Fish/Preplacode/tiles
Patch size: 1024, overlap: 0.25

GPUs: 3
Output: <run>/WT_Developing_Fish/Preplacode/Preplacode


## 9. Build & Run Pipeline

In [ ]:
from packages.epicycle_scaledseg.core.pipeline import EpiCyclePipeline

pipeline = EpiCyclePipeline(
    work_dir=str(work_dir),
    n_gpus=n_gpus,
    log_dir=str(log_dir),
)

# --- Stage 0: Tiling ---
if RUN_TILING:
    from packages.epicycle_scaledseg.core.tiling import TilingStage
    pipeline.add_stage(TilingStage(
        input_tif=INPUT_TIF,
        output_dir=str(tiles_dir),
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        z_threshold_method=Z_THRESHOLD_METHOD,
        threshold_value=Z_THRESHOLD_VALUE,
        channel_index=CHANNEL_INDEX,
        auto_correct=AUTO_CORRECT_PATCH,
    ))
    # Tiling creates tile_dirs; use placeholder for now
    tile_dirs = [str(work_dir)]

# --- Stage 1: Segmentation ---
if RUN_SEGMENTATION:
    from packages.epicycle_scaledseg.core.segmentation import SegmentationStage
    pipeline.add_stage(SegmentationStage(
        lateral_model=LATERAL_MODEL,
        axial_model=AXIAL_MODEL,
        out_root=str(output_dir / "stage1_segmentation"),
        gpu_strategy=GPU_STRATEGY,
    ))

# --- Stage 2: Reconstruction ---
if RUN_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.reconstruction import ReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_reconstruct_params
    reconstruct_params = get_default_reconstruct_params()
    reconstruct_params['gradient_descent']['renorm_gradient'] = RECON_RENORM_GRADIENT
    pipeline.add_stage(ReconstructionStage(
        seg_root=str(output_dir / "stage1_segmentation"),
        out_root=str(output_dir / "stage2_reconstruction"),
        mode=RECON_MODE,
        reconstruct_params=reconstruct_params,
        max_workers=RECON_MAX_WORKERS,
        n_cpu_per_worker=N_CPU_PER_WORKER,
        save_prob3d=SAVE_PROB3D,
        save_flow3d=SAVE_FLOW3D,
        save_binary=SAVE_BINARY,
        binary_erode=BINARY_ERODE,
        binary_kernel_size=BINARY_KERNEL_SIZE,
        binary_border_margin=BINARY_BORDER_MARGIN,
        binary_fill_holes=BINARY_FILL_HOLES,
        cleanup_intermediates=CLEANUP_INTERMEDIATES,
    ))

# --- Stage 3: Assembly ---
if RUN_ASSEMBLY:
    from packages.epicycle_scaledseg.core.assembly import AssemblyStage
    pipeline.add_stage(AssemblyStage(
        recon_root=str(output_dir / "stage2_reconstruction"),
        output_dir=str(output_dir / "stage3_assembly"),
        full_size=FULL_SIZE,
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        depth=DEPTH,
        stitch_mask=STITCH_MASK,
        stitch_prob=STITCH_PROB and SAVE_PROB3D,
        stitch_binary=STITCH_BINARY and SAVE_BINARY,
        prob_stitch_mode=PROB_STITCH_MODE,
    ))

# --- Stage 4: Global Indirect Reconstruction ---
if RUN_GLOBAL_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.global_reconstruction import GlobalReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_indirect_reconstruct_params

    indirect_params = get_default_indirect_reconstruct_params(n_cpu=GLOBAL_N_CPU)
    indirect_params['gradient_descent']['n_iter'] = GLOBAL_N_ITER
    indirect_params['gradient_descent']['momenta'] = GLOBAL_MOMENTA
    indirect_params['gradient_descent']['gradient_decay'] = GLOBAL_GRADIENT_DECAY
    indirect_params['gradient_descent']['renorm_gradient'] = GLOBAL_RENORM_GRADIENT
    indirect_params['indirect_method']['dtform_method'] = GLOBAL_DTFORM_METHOD
    indirect_params['indirect_method']['smooth_skel_sigma'] = GLOBAL_SMOOTH_SKEL_SIGMA
    if GLOBAL_TILE_SHAPE is not None:
        indirect_params['gradient_descent']['tile_shape'] = GLOBAL_TILE_SHAPE

    pipeline.add_stage(GlobalReconstructionStage(
        assembly_dir=str(output_dir / "stage3_assembly"),
        output_dir=str(output_dir / "stage4_global"),
        downsample_factor=DOWNSAMPLE_FACTOR,
        min_size=GLOBAL_MIN_SIZE,
        use_precomputed_binary=USE_PRECOMPUTED_BINARY,
        indirect_params=indirect_params,
        n_cpu=GLOBAL_N_CPU,
        tile_shape=GLOBAL_TILE_SHAPE,
    ))

print(f"Pipeline: {len(pipeline.stages)} stages")
for s in pipeline.stages:
    print(f"  - {s.name}")
print(f"Tile dirs: {len(tile_dirs)}")
print(f"Output: {output_dir}")

# ===== RUN =====
result = pipeline.run(tile_dirs)
print()
print(result.summary())

2026-02-04 13:38:35,633 epicycle.pipeline INFO Pipeline starting: 5 stage(s), 1 tile(s), 3 GPU(s)
2026-02-04 13:38:35,634 epicycle.pipeline INFO Run parameters saved to data/WT_Developing_Fish/Preplacode/Preplacode/logs/run_params.json
2026-02-04 13:38:35,635 epicycle.pipeline INFO === Stage: tiling ===
2026-02-04 13:38:35,636 epicycle.tiling INFO Stage 'tiling' starting with 1 tile(s)
2026-02-04 13:38:35,637 epicycle.tiling INFO Parameters: {"input_tif": "data/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL7.7_PREplacode_002 - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif", "patch_size": 1024, "overlap": 0.25, "z_threshold_method": "hard", "threshold_value": 0.07, "channel_index": 1, "auto_correct": false}
2026-02-04 13:38:35,637 epicycle.core.tiling INFO Tiling data/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL7.7_PREplacode_002 - Stitched - Denoised.nd2-PROCESSED-ISO-1792x1792.tif -> data/WT_Developing_Fish/Preplacode/tiles (patch=1024, overlap=0.25

Pipeline: 5 stages
  - tiling
  - segmentation
  - reconstruction
  - assembly
  - global_reconstruction
Tile dirs: 1
Output: <run>/WT_Developing_Fish/Preplacode/Preplacode


2026-02-04 13:38:37,955 epicycle.utils.tiling INFO Z-stack shape: Z=371, C=2, H=1792, W=1792, dtype=uint16
2026-02-04 13:38:37,957 epicycle.utils.tiling INFO Tiling: patch_size=1024, step=768, overlap=0.25
2026-02-04 13:38:37,957 epicycle.utils.tiling INFO Grid: 2 rows x 2 cols = 4 tiles
Saving tiles: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.30tile/s]
2026-02-04 13:38:41,049 epicycle.core.tiling INFO Tiling complete: 4 tiles, actual patch=1024, step=768
2026-02-04 13:38:41,058 epicycle.pipeline INFO Stage 'tiling': 4/4 succeeded (5.4s)
2026-02-04 13:38:41,059 epicycle.tiling INFO Stage 'tiling' finished: 4/4 succeeded (5.4s)
2026-02-04 13:38:41,059 epicycle.pipeline INFO Stage 'tiling' produced 4 tile directories
2026-02-04 13:38:41,060 epicycle.pipeline INFO === Stage: segmentation ===
2026-02-04 13:38:41,060 epicycle.segmentation INFO Stage 'segmentation' starting with 4 tile(s)
2026

Automatric binary thresholding:  0.6
[ 277 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  108
start dynamics
Tile id 30 62331 62331
Tile id 31 95219 95219
Tile id 32 233030 233030
Tile id 33 304601 304601
Tile id 18 322288 322288
Tile id 24 383006 383006
Tile id 25 400911 400911
Tile id 35 427927 427927
Tile id 0 398128 398128
Tile id 34 490467 490467
Tile id 13 506733 506733
Tile id 41 517157 517157
Tile id 12 563367 563367
Tile id 26 595944 595944
Tile id 19 590161 590161
Tile id 6 637155 637155
Tile id 75 0 0
Tile id 20 650794 650794
Tile id 76 0 0
Tile id 77 0 0
Tile id 72 36172 36172
Tile id 74 31893 31893
Tile id 73 53837 53837
Tile id 82 677 677
Tile id 83 0 0
Tile id 17 864640 864640
Tile id 27 860382 860382
Tile id 81 23393 23393
Tile id 21 879097 879097
Tile id 29 918260 918260
Tile id 23 935238 935238
Tile id 40 913760 913760
Tile id 5 1003335 1003335
Tile id 1 1022362 1022362
Tile id 80 187150 187150
Tile id 7 1050408 1050408
Tile id 14 1072414 107

Eroding (GPU): 100%|██████████| 1189/1189 [00:07<00:00, 157.25inst/s]


Automatric binary thresholding:  0.6
[ 309 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  108
start dynamics
Tile id 32 8626 8626
Tile id 33 30917 30917
Tile id 27 32573 32573
Tile id 35 37978 37978
Tile id 12 43408 43408
Tile id 34 55487 55487
Tile id 20 74138 74138
Tile id 21 96528 96528
Tile id 26 103419 103419
Tile id 30 103540 103540
Tile id 13 132193 132193
Tile id 19 148696 148696
Tile id 31 154452 154452
Tile id 24 156762 156762
Tile id 28 199067 199067
Tile id 18 220533 220533
Tile id 25 208497 208497
Tile id 74 324 324
Tile id 75 0 0
Tile id 76 0 0
Tile id 41 274136 274136
Tile id 77 0 0
Tile id 6 334526 334526
Tile id 29 356871 356871
Tile id 72 47931 47931
Tile id 73 27807 27807
Tile id 83 0 0
Tile id 14 434450 434450
Tile id 82 6318 6318
Tile id 81 34703 34703
Tile id 0 542676 542676
Tile id 80 168063 168063
Tile id 40 688742 688742
Tile id 22 686011 686011
Tile id 23 720328 720328
Tile id 5 713320 713320
Tile id 39 762516 762516
Tile id 78 399532

... (4 outputs left out)

Eroding (GPU): 100%|██████████| 1069/1069 [00:05<00:00, 186.87inst/s]
2026-02-04 13:54:48,570 epicycle.pipeline INFO Stage 'reconstruction': 4/4 succeeded (517.7s)
2026-02-04 13:54:48,571 epicycle.reconstruction INFO Stage 'reconstruction' finished: 4/4 succeeded (517.7s)
2026-02-04 13:54:48,571 epicycle.pipeline INFO === Stage: assembly ===
2026-02-04 13:54:48,572 epicycle.assembly INFO Stage 'assembly' starting with 4 tile(s)
2026-02-04 13:54:48,573 epicycle.assembly INFO Parameters: {"full_size": [1792, 1792], "tile_size": 1024, "overlap": 0.25, "depth": null, "stitch_mask": true, "stitch_prob": true, "stitch_binary": true, "prob_stitch_mode": "max", "binary_format": "npy", "n_workers": 16}
2026-02-04 13:54:48,573 epicycle.assembly INFO Loading tiling config from data/WT_Developing_Fish/Preplacode/tiles/tiling_config.json
2026-02-04 13:54:48,573 epicycle.assembly INFO Loading tiling config from data/WT_Developing_Fish/Preplacode/tiles/tiling_config.json
2026-02-04 13:54:48,574 epicy

[ 281 1088 1088] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  108
start dynamics
Tile id 35 16797 16797
Tile id 23 13759 13759
Tile id 29 43554 43554
Tile id 17 50593 50593
Tile id 0 23129 23129
Tile id 41 64514 64514
Tile id 12 31504 31504
Tile id 6 43895 43895
Tile id 30 87973 87973
Tile id 5 88821 88821
Tile id 13 58102 58102
Tile id 34 113360 113360
Tile id 11 110481 110481
Tile id 15 80083 80083
Tile id 21 82613 82613
Tile id 33 123953 123953
Tile id 22 84571 84571
Tile id 7 96004 96004
Tile id 72 2 2
Tile id 73 0 0
Tile id 77 0 0
Tile id 74 0 0
Tile id 75 0 0
Tile id 31 169784 169784
Tile id 76 0 0
Tile id 32 178444 178444
Tile id 83 0 0
Tile idTile id 20 155178 155178
 80 0 0
Tile id 59 196244 196244
Tile id 47 191239 191239
Tile id 78 248 248
Tile id 1 161066 161066
Tile id 81 0 0
Tile id 82 0 0
Tile id 16 166187 166187
Tile id 8 171448 171448
Tile id 89 12 12
Tile id 79 513 513
Tile id 53 224158 224158
Tile id 14 199453 199453
Tile id 95 297 297
Tile id 101 0 

2026-02-04 13:59:14,966 epicycle.core.global_reconstruction INFO Global reconstruction complete: shape=(185, 896, 896), n_labels=2493, elapsed=230.5s
2026-02-04 13:59:14,967 epicycle.core.global_reconstruction INFO Saving outputs to <run>/WT_Developing_Fish/Preplacode/Preplacode/stage4_global
2026-02-04 13:59:28,460 epicycle.pipeline INFO Stage 'global_reconstruction': 1/1 succeeded (249.0s)
2026-02-04 13:59:28,461 epicycle.global_reconstruction INFO Stage 'global_reconstruction' finished: 1/1 succeeded (249.0s)
2026-02-04 13:59:28,462 epicycle.pipeline WARNING No tiles remaining — stopping pipeline.
2026-02-04 13:59:28,462 epicycle.pipeline INFO Pipeline: 1/4 tiles (1252.8s total)
  tiling: 4/4 succeeded (5.4s)
  segmentation: 4/4 succeeded (449.8s)
  reconstruction: 4/4 succeeded (517.7s)
  assembly: 4/4 succeeded (30.9s)
  global_reconstruction: 1/1 succeeded (249.0s)



Pipeline: 1/4 tiles (1252.8s total)
  tiling: 4/4 succeeded (5.4s)
  segmentation: 4/4 succeeded (449.8s)
  reconstruction: 4/4 succeeded (517.7s)
  assembly: 4/4 succeeded (30.9s)
  global_reconstruction: 1/1 succeeded (249.0s)


## 10. Check Status

In [ ]:
# Quick status check — how many tiles completed each stage?
seg_root = output_dir / "stage1_segmentation"
recon_root = output_dir / "stage2_reconstruction"

seg_done = sum(1 for td in tile_dirs if (seg_root / Path(td).name / "SEGMENTATION_DONE").exists())
recon_done = sum(1 for td in tile_dirs if (recon_root / Path(td).name / f"{Path(td).name}_mask.tif").exists())

print(f"Stage 1 (Segmentation):   {seg_done}/{len(tile_dirs)} complete")
print(f"Stage 2 (Reconstruction): {recon_done}/{len(tile_dirs)} complete")
print(f"Stage 3 (Assembly):       check {output_dir / 'stage3_assembly'}")
print(f"Stage 4 (Global Recon):   {'done' if (output_dir / 'stage4_global' / 'global_segmentation.tif').exists() else 'not done'}")

Stage 1 (Segmentation):   0/1 complete
Stage 2 (Reconstruction): 0/1 complete
Stage 3 (Assembly):       check <run>/WT_Developing_Fish/Preplacode/Preplacode/stage3_assembly
Stage 4 (Global Recon):   done
